In [1]:
%pip install -q numpy pandas rasterio geopandas requests folium scikit-learn matplotlib seaborn tqdm earthpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 71.6 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
import geopandas as gpd
from shapely.geometry import box, Point
import requests
from datetime import datetime, timedelta
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import folium
from folium import plugins
import warnings
warnings.filterwarnings('ignore')

In [3]:
class WildfireDetectionSystem:
    """Main system for wildfire detection using satellite thermal data"""

    def __init__(self, output_dir='wildfire_data'):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.data_dir = self.output_dir / 'raw_data'
        self.data_dir.mkdir(exist_ok=True)
        self.model = None
        self.scaler = StandardScaler()

    def download_modis_firms_data(self, country='USA', days_back=7):
        """
        Download MODIS/VIIRS active fire data from NASA FIRMS
        Register for free API key at: https://firms.modaps.eosdis.nasa.gov/api/
        """
        print(f"Downloading FIRMS data for past {days_back} days...")

        # Using the public FIRMS data feed (no API key needed for recent data)
        # For production, register for API key at FIRMS website
        base_url = "https://firms.modaps.eosdis.nasa.gov/data/active_fire/"

        # For demonstration, we'll create synthetic data
        # In production, replace with actual API calls
        df = self._generate_synthetic_fire_data(country, days_back)

        output_file = self.data_dir / f'firms_data_{country}_{days_back}days.csv'
        df.to_csv(output_file, index=False)
        print(f"Data saved to {output_file}")
        return df

    def _generate_synthetic_fire_data(self, country, days_back):
        """Generate synthetic fire detection data for demonstration"""
        np.random.seed(42)

        # Simulate fire hotspots in California region
        n_samples = 1000

        # Normal background readings
        normal_lat = np.random.normal(37.5, 2.0, int(n_samples * 0.85))
        normal_lon = np.random.normal(-120.0, 2.0, int(n_samples * 0.85))
        normal_brightness = np.random.normal(300, 15, int(n_samples * 0.85))
        normal_frp = np.random.exponential(5, int(n_samples * 0.85))

        # Fire hotspots
        fire_lat = np.random.normal(38.5, 0.3, int(n_samples * 0.15))
        fire_lon = np.random.normal(-121.0, 0.3, int(n_samples * 0.15))
        fire_brightness = np.random.normal(350, 25, int(n_samples * 0.15))
        fire_frp = np.random.exponential(50, int(n_samples * 0.15))

        lat = np.concatenate([normal_lat, fire_lat])
        lon = np.concatenate([normal_lon, fire_lon])
        brightness = np.concatenate([normal_brightness, fire_brightness])
        frp = np.concatenate([normal_frp, fire_frp])

        # Add temporal variation
        dates = [datetime.now() - timedelta(days=np.random.randint(0, days_back))
                 for _ in range(n_samples)]

        df = pd.DataFrame({
            'latitude': lat,
            'longitude': lon,
            'brightness': brightness,
            'bright_t31': brightness + np.random.normal(0, 5, n_samples),
            'frp': frp,  # Fire Radiative Power
            'confidence': np.random.choice(['l', 'n', 'h'], n_samples, p=[0.2, 0.5, 0.3]),
            'acq_date': [d.strftime('%Y-%m-%d') for d in dates],
            'acq_time': [f"{np.random.randint(0, 24):02d}{np.random.randint(0, 60):02d}" for _ in range(n_samples)],
            'satellite': np.random.choice(['Terra', 'Aqua'], n_samples)
        })

        return df

    def preprocess_data(self, df):
        """
        Preprocess satellite data:
        - Handle missing values
        - Remove cloud contamination
        - Calibrate brightness temperature
        - Feature engineering
        """
        print("Preprocessing data...")

        # Convert confidence to numeric
        confidence_map = {'l': 0, 'n': 1, 'h': 2}
        df['confidence_numeric'] = df['confidence'].map(confidence_map)

        # Remove low confidence detections
        df = df[df['confidence'] != 'l'].copy()

        # Feature engineering
        df['brightness_diff'] = df['brightness'] - df['bright_t31']
        df['brightness_ratio'] = df['brightness'] / (df['bright_t31'] + 1)
        df['log_frp'] = np.log1p(df['frp'])

        # Temporal features
        df['acq_datetime'] = pd.to_datetime(df['acq_date'] + ' ' + df['acq_time'],
                                            format='%Y-%m-%d %H%M')
        df['hour'] = df['acq_datetime'].dt.hour
        df['day_of_week'] = df['acq_datetime'].dt.dayofweek

        # Spatial features (simplified grid-based)
        df['lat_rounded'] = df['latitude'].round(2)
        df['lon_rounded'] = df['longitude'].round(2)

        # Count detections per grid cell (temporal density)
        grid_counts = df.groupby(['lat_rounded', 'lon_rounded']).size().reset_index(name='grid_count')
        df = df.merge(grid_counts, on=['lat_rounded', 'lon_rounded'], how='left')

        print(f"Preprocessed data shape: {df.shape}")
        return df

    def create_labels(self, df, brightness_threshold=330, frp_threshold=20):
        """
        Create ground truth labels for training
        In production, use validated fire perimeters from agencies
        """
        # Simple rule-based labeling (replace with actual labels in production)
        df['is_fire'] = (
            (df['brightness'] > brightness_threshold) &
            (df['frp'] > frp_threshold) &
            (df['confidence'] == 'h')
        ).astype(int)

        print(f"Fire detections: {df['is_fire'].sum()} / {len(df)}")
        return df

    def build_anomaly_detector(self, df):
        """Build Isolation Forest for anomaly detection (unsupervised)"""
        print("Training Isolation Forest anomaly detector...")

        feature_cols = ['brightness', 'bright_t31', 'frp', 'log_frp',
                       'brightness_diff', 'confidence_numeric', 'grid_count']
        X = df[feature_cols].fillna(0)

        # Standardize features
        X_scaled = self.scaler.fit_transform(X)

        # Train Isolation Forest
        iso_forest = IsolationForest(
            contamination=0.15,  # Expected proportion of anomalies
            random_state=42,
            n_estimators=100
        )

        # Predict (-1 for anomalies, 1 for normal)
        predictions = iso_forest.fit_predict(X_scaled)
        df['anomaly_score'] = iso_forest.score_samples(X_scaled)
        df['is_anomaly'] = (predictions == -1).astype(int)

        print(f"Detected anomalies: {df['is_anomaly'].sum()}")
        return df, iso_forest

    def build_supervised_classifier(self, df):
        """Build Random Forest classifier (supervised)"""
        print("Training Random Forest classifier...")

        feature_cols = ['brightness', 'bright_t31', 'frp', 'log_frp',
                       'brightness_diff', 'brightness_ratio',
                       'confidence_numeric', 'grid_count', 'hour']

        X = df[feature_cols].fillna(0)
        y = df['is_fire']

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        # Standardize
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        # Train classifier
        rf_model = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            class_weight='balanced'
        )
        rf_model.fit(X_train_scaled, y_train)

        # Evaluate
        y_pred = rf_model.predict(X_test_scaled)
        y_pred_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))

        # Feature importance
        feature_importance = pd.DataFrame({
            'feature': feature_cols,
            'importance': rf_model.feature_importances_
        }).sort_values('importance', ascending=False)

        print("\nTop 5 Important Features:")
        print(feature_importance.head())

        # Store predictions
        df.loc[X_test.index, 'fire_probability'] = y_pred_proba

        self.model = rf_model
        return df, rf_model, feature_importance

    def visualize_detections(self, df, method='supervised'):
        """Create static visualizations"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))

        # Detection map
        ax1 = axes[0, 0]
        if method == 'supervised':
            fire_points = df[df['is_fire'] == 1]
            normal_points = df[df['is_fire'] == 0]
            ax1.scatter(normal_points['longitude'], normal_points['latitude'],
                       c='blue', alpha=0.3, s=10, label='Normal')
            ax1.scatter(fire_points['longitude'], fire_points['latitude'],
                       c='red', alpha=0.7, s=30, label='Fire')
        else:
            anomaly_points = df[df['is_anomaly'] == 1]
            normal_points = df[df['is_anomaly'] == 0]
            ax1.scatter(normal_points['longitude'], normal_points['latitude'],
                       c='blue', alpha=0.3, s=10, label='Normal')
            ax1.scatter(anomaly_points['longitude'], anomaly_points['latitude'],
                       c='red', alpha=0.7, s=30, label='Anomaly')

        ax1.set_xlabel('Longitude')
        ax1.set_ylabel('Latitude')
        ax1.set_title('Wildfire Detection Map')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # Brightness distribution
        ax2 = axes[0, 1]
        if method == 'supervised':
            ax2.hist(df[df['is_fire']==0]['brightness'], bins=50, alpha=0.5, label='Normal')
            ax2.hist(df[df['is_fire']==1]['brightness'], bins=50, alpha=0.5, label='Fire')
        else:
            ax2.hist(df[df['is_anomaly']==0]['brightness'], bins=50, alpha=0.5, label='Normal')
            ax2.hist(df[df['is_anomaly']==1]['brightness'], bins=50, alpha=0.5, label='Anomaly')
        ax2.set_xlabel('Brightness Temperature (K)')
        ax2.set_ylabel('Count')
        ax2.set_title('Brightness Temperature Distribution')
        ax2.legend()

        # FRP distribution
        ax3 = axes[1, 0]
        if method == 'supervised':
            ax3.hist(df[df['is_fire']==0]['frp'], bins=50, alpha=0.5, label='Normal')
            ax3.hist(df[df['is_fire']==1]['frp'], bins=50, alpha=0.5, label='Fire')
        else:
            ax3.hist(df[df['is_anomaly']==0]['frp'], bins=50, alpha=0.5, label='Normal')
            ax3.hist(df[df['is_anomaly']==1]['frp'], bins=50, alpha=0.5, label='Anomaly')
        ax3.set_xlabel('Fire Radiative Power (MW)')
        ax3.set_ylabel('Count')
        ax3.set_title('Fire Radiative Power Distribution')
        ax3.set_xlim(0, 200)
        ax3.legend()

        # Temporal pattern
        ax4 = axes[1, 1]
        hourly_fires = df[df.get('is_fire', df.get('is_anomaly', 0)) == 1].groupby('hour').size()
        ax4.bar(hourly_fires.index, hourly_fires.values, color='orangered')
        ax4.set_xlabel('Hour of Day')
        ax4.set_ylabel('Number of Detections')
        ax4.set_title('Fire Detections by Hour')
        ax4.set_xticks(range(0, 24, 3))

        plt.tight_layout()
        output_file = self.output_dir / f'detection_analysis_{method}.png'
        plt.savefig(output_file, dpi=300, bbox_inches='tight')
        print(f"Visualization saved to {output_file}")
        plt.close()

    def create_interactive_map(self, df, method='supervised', output_file='wildfire_map.html'):
        """Create interactive Folium map with fire detections"""
        print("Creating interactive map...")

        # Center map on data
        center_lat = df['latitude'].mean()
        center_lon = df['longitude'].mean()

        m = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=7,
            tiles='OpenStreetMap'
        )

        # Add different tile layers
        folium.TileLayer('CartoDB dark_matter', name='Dark Mode').add_to(m)

        # Create feature groups
        fire_group = folium.FeatureGroup(name='Fire Detections')
        normal_group = folium.FeatureGroup(name='Normal Readings')

        # Add markers
        if method == 'supervised':
            fire_df = df[df['is_fire'] == 1]
            normal_df = df[df['is_fire'] == 0].sample(min(100, len(df[df['is_fire']==0])))
        else:
            fire_df = df[df['is_anomaly'] == 1]
            normal_df = df[df['is_anomaly'] == 0].sample(min(100, len(df[df['is_anomaly']==0])))

        # Add fire markers
        for idx, row in fire_df.iterrows():
            popup_text = f"""
            <b>Fire Detection</b><br>
            Date: {row['acq_date']}<br>
            Time: {row['acq_time']}<br>
            Brightness: {row['brightness']:.1f}K<br>
            FRP: {row['frp']:.1f} MW<br>
            Confidence: {row['confidence']}<br>
            Satellite: {row['satellite']}
            """

            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=5 + row['frp']/20,  # Size based on FRP
                popup=folium.Popup(popup_text, max_width=200),
                color='darkred',
                fill=True,
                fillColor='orangered',
                fillOpacity=0.7,
                weight=2
            ).add_to(fire_group)

        # Add normal markers (sample)
        for idx, row in normal_df.iterrows():
            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=2,
                popup=f"Normal reading: {row['brightness']:.1f}K",
                color='blue',
                fill=True,
                fillColor='lightblue',
                fillOpacity=0.3,
                weight=1
            ).add_to(normal_group)

        # Add heatmap for fire intensity
        heat_data = [[row['latitude'], row['longitude'], row['frp']]
                     for idx, row in fire_df.iterrows()]
        plugins.HeatMap(heat_data, name='Fire Intensity Heatmap',
                       min_opacity=0.4, radius=15).add_to(m)

        # Add groups to map
        fire_group.add_to(m)
        normal_group.add_to(m)

        # Add layer control
        folium.LayerControl().add_to(m)

        # Add legend
        legend_html = '''
        <div style="position: fixed; bottom: 50px; left: 50px; width: 200px;
        background-color: white; border:2px solid grey; z-index:9999; font-size:14px;
        padding: 10px">
        <p><b>Legend</b></p>
        <p><span style="color:orangered;">●</span> Fire Detection</p>
        <p><span style="color:lightblue;">●</span> Normal Reading</p>
        </div>
        '''
        m.get_root().html.add_child(folium.Element(legend_html))

        # Save map
        output_path = self.output_dir / output_file
        m.save(str(output_path))
        print(f"Interactive map saved to {output_path}")
        return m

    def create_temporal_animation_data(self, df, output_file='fire_temporal.json'):
        """Export data for temporal animation (can be used with deck.gl or similar)"""
        print("Creating temporal animation data...")

        fire_df = df[df.get('is_fire', df.get('is_anomaly', 0)) == 1].copy()
        fire_df['timestamp'] = pd.to_datetime(fire_df['acq_datetime']).astype(int) // 10**9

        animation_data = []
        for idx, row in fire_df.iterrows():
            animation_data.append({
                'lat': float(row['latitude']),
                'lon': float(row['longitude']),
                'timestamp': int(row['timestamp']),
                'brightness': float(row['brightness']),
                'frp': float(row['frp']),
                'date': row['acq_date']
            })

        output_path = self.output_dir / output_file
        with open(output_path, 'w') as f:
            json.dump(animation_data, f)

        print(f"Temporal data exported to {output_path}")
        return animation_data

In [4]:
# Example usage pipeline
def main():
    """Run complete wildfire detection pipeline"""

    print("=" * 60)
    print("Wildfire Detection & Mapping System")
    print("=" * 60)

    # Initialize system
    system = WildfireDetectionSystem(output_dir='wildfire_results')

    # Step 1: Download data
    df = system.download_modis_firms_data(country='USA', days_back=7)

    # Step 2: Preprocess
    df = system.preprocess_data(df)

    # Step 3: Create labels (for supervised learning)
    df = system.create_labels(df, brightness_threshold=330, frp_threshold=20)

    # Step 4a: Unsupervised anomaly detection
    print("\n" + "="*60)
    print("Running Unsupervised Anomaly Detection")
    print("="*60)
    df, iso_model = system.build_anomaly_detector(df)
    system.visualize_detections(df, method='unsupervised')
    system.create_interactive_map(df, method='unsupervised',
                                  output_file='map_anomaly.html')

    # Step 4b: Supervised classification
    print("\n" + "="*60)
    print("Running Supervised Classification")
    print("="*60)
    df, rf_model, importance = system.build_supervised_classifier(df)
    system.visualize_detections(df, method='supervised')
    system.create_interactive_map(df, method='supervised',
                                  output_file='map_supervised.html')

    # Step 5: Export temporal data
    system.create_temporal_animation_data(df)

    print("\n" + "="*60)
    print("Pipeline Complete!")
    print("="*60)
    print(f"Results saved to: {system.output_dir}")
    print("- Interactive maps: map_anomaly.html, map_supervised.html")
    print("- Visualizations: detection_analysis_*.png")
    print("- Temporal data: fire_temporal.json")

    return system, df

if __name__ == "__main__":
    system, df = main()

Wildfire Detection & Mapping System
Data saved to wildfire_results/raw_data/firms_data_USA_7days.csv
Preprocessing data...
Preprocessed data shape: (803, 19)
Fire detections: 28 / 803

Running Unsupervised Anomaly Detection
Training Isolation Forest anomaly detector...
Detected anomalies: 121
Visualization saved to wildfire_results/detection_analysis_unsupervised.png
Creating interactive map...
Interactive map saved to wildfire_results/map_anomaly.html

Running Supervised Classification
Training Random Forest classifier...

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       155
           1       1.00      0.67      0.80         6

    accuracy                           0.99       161
   macro avg       0.99      0.83      0.90       161
weighted avg       0.99      0.99      0.99       161


Top 5 Important Features:
              feature  importance
0          brightness    0.249002
1          bright_t31    